# 02 — Model Evaluation

This notebook runs a full evaluation of the trained model:
1. **Perplexity** on held-out text
2. **Generation quality** — repetition, diversity, length
3. **Failure mode analysis** — repetition, truncation, topic drift
4. **Sample generations** — story writing and text completion
5. **(Optional) LLM-as-judge** — scoring via Ollama

Run `python src/training/pretrain.py` and `python src/training/finetune.py` first to create checkpoints.

In [ ]:
import sys, os, json
sys.path.insert(0, '..')

import torch
import matplotlib.pyplot as plt

from src.demo.interface import GenerationInterface
from src.evaluation.metrics import compute_perplexity, evaluate_generations, print_metrics_table
from src.evaluation.error_analysis import analyse_batch, print_error_report
from src.tokenizer.bpe_tokenizer import BPETokenizer

# ── Load model ──────────────────────────────────────────────────────
iface = GenerationInterface()   # loads best available checkpoint
print(iface.model)

## 1. Perplexity

In [ ]:
tokenizer = BPETokenizer()

with open('../data/pretrain/data.txt', encoding='utf-8') as f:
    text = f.read()

# Use last 10% as held-out evaluation set
val_text = text[int(len(text) * 0.9):]

ppl = compute_perplexity(
    val_text,
    iface.model,
    tokenizer,
    iface.device,
    max_length=iface.context_size,
)

print(f'Validation Perplexity: {ppl:.2f}')
print()
print('Interpretation:')
print('  < 20   → Excellent (possible overfitting)')
print('  20-80  → Good for a small model')
print('  80-200 → Acceptable (needs more training/data)')
print('  > 200  → Poor (underfitting)')

## 2. Generation quality metrics

In [ ]:
story_prompts = [
    'The lighthouse keeper walked to the cliff edge and',
    'She found the letter in her mother\'s coat and',
    'The detective arrived at the museum just after dawn.',
    'After thirty years, he returned to the village where',
    'The market in the old quarter smelled of cardamom and',
    'Old Thomas had been keeping the lighthouse for',
    'She learned carpentry from her great-uncle in a shed',
    'The twins had not spoken to each other since',
]

metrics = evaluate_generations(
    iface.model, tokenizer, iface.device,
    story_prompts,
    max_tokens=120, temperature=0.8, top_k=40,
)

print_metrics_table({k: v for k, v in metrics.items() if k != 'samples'})

## 3. Sample generations

In [ ]:
for i, (prompt, generated) in enumerate(zip(story_prompts, metrics['samples']), 1):
    print(f'\n──── Sample {i} ────────────────────────────────────────')
    print(f'Prompt   : {prompt}')
    print(f'Generated: {generated}')

## 4. Instruction-following samples

In [ ]:
instructions = [
    ('Write a short story about a lighthouse keeper.', ''),
    ('Write a haiku about the sea at dawn.', ''),
    ('Write a story about two friends reuniting after years apart.', ''),
    ('Continue this story opening.', 'The map was wrong, which was how they found the village.'),
    ('Write a poem about learning something new late in life.', ''),
]

for instruction, context in instructions:
    response, stats = iface.instruct(instruction, context, max_tokens=200, temperature=0.75)
    print(f'\n{'='*60}')
    print(f'Instruction: {instruction}')
    if context:
        print(f'Context    : {context}')
    print(f'\nResponse ({stats["new_tokens"]} tokens):')
    print(response)

## 5. Failure mode analysis

In [ ]:
analysis = analyse_batch(story_prompts, metrics['samples'])
print_error_report(analysis)

print('\nIndividual sample breakdown:')
for i, result in enumerate(analysis['individual'], 1):
    flags = [name for name, r in result.items() if r.get('detected')]
    status = ', '.join(flags) if flags else 'clean'
    print(f'  Sample {i}: {status}')

## 6. Temperature sensitivity

In [ ]:
# How does temperature affect quality vs. diversity?
from src.evaluation.metrics import repetition_rate, distinct_n

temperatures = [0.3, 0.5, 0.7, 0.9, 1.1]
test_prompt  = 'The lighthouse keeper walked to the cliff and'

rep_rates, dist_scores = [], []
for temp in temperatures:
    texts = []
    for _ in range(5):
        full, _ = iface.complete(test_prompt, max_tokens=80, temperature=temp, top_k=40)
        texts.append(full[len(test_prompt):])
    rep_rates.append(sum(repetition_rate(t) for t in texts) / len(texts))
    dist_scores.append(distinct_n(texts, 2))

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(temperatures, rep_rates, 'o-', color='#dc2626')
axes[0].set_title('Repetition Rate vs Temperature')
axes[0].set_xlabel('Temperature')
axes[0].set_ylabel('Repetition (lower=better)')

axes[1].plot(temperatures, dist_scores, 'o-', color='#2563eb')
axes[1].set_title('Diversity (Distinct-2) vs Temperature')
axes[1].set_xlabel('Temperature')
axes[1].set_ylabel('Distinct-2 (higher=better)')

plt.tight_layout()
plt.savefig('../results/plots/temperature_analysis.png', dpi=150)
plt.show()
print('Saved → results/plots/temperature_analysis.png')

## 7. (Optional) LLM-as-judge evaluation

Requires Ollama running locally:
```bash
ollama serve
ollama pull llama3
```

In [ ]:
# Uncomment to run LLM-as-judge evaluation

# from src.evaluation.human_eval import score_all, print_judge_report, is_ollama_running
#
# if is_ollama_running():
#     sft_data = json.load(open('../data/finetune/story_completion/data.json'))
#
#     # Generate model responses
#     eval_entries = []
#     for entry in sft_data[:15]:
#         response, _ = iface.instruct(entry['instruction'], entry.get('input',''))
#         eval_entries.append({**entry, 'model_response': response})
#
#     scores, avg = score_all(eval_entries, verbose=True)
#     print_judge_report(scores, avg)
#
#     # Save
#     with open('../results/sample_generations/llm_judge_results.json', 'w') as f:
#         json.dump(eval_entries, f, indent=2)
# else:
#     print('Ollama not running. Skipping LLM judge evaluation.')

print('LLM judge cell ready — uncomment to run.')

## 8. Save evaluation report

In [ ]:
import json, os
os.makedirs('../results/sample_generations', exist_ok=True)
os.makedirs('../results/plots', exist_ok=True)

report = {
    'perplexity': round(ppl, 2),
    'generation_metrics': {k: v for k, v in metrics.items() if k != 'samples'},
    'failure_mode_counts': analysis['failure_counts'],
    'low_diversity': analysis['low_diversity'],
    'sample_generations': [
        {'prompt': p, 'generated': g}
        for p, g in zip(story_prompts, metrics['samples'])
    ],
}

with open('../results/evaluation_report.json', 'w') as f:
    json.dump(report, f, indent=2)

print('Evaluation report saved to results/evaluation_report.json')
print(f'\nSummary:')
print(f'  Perplexity  : {ppl:.2f}')
print(f'  Repetition  : {metrics["repetition"]:.4f}')
print(f'  Distinct-2  : {metrics["distinct_2"]:.4f}')
print(f'  Avg words   : {metrics["avg_words"]:.1f}')
print(f'  Proper end  : {metrics["pct_proper_end"]}%')